# Branching Particle Filter — Sandbox

Interactive notebook for invoking and visualising the **branching particle filter baseline** (model spec §9.2, build step 4).

**Kernel**: Use the `vae-env-cluster` environment.

---

**Sections**:
1. Setup & data loading
2. Instantiate the particle filter
3. Layer 1: reference selection (inspect weights & speed ratios)
4. Layer 2+3: forward prediction with recruitment (single example)
5. Prediction fan: visualise the weighted particle cloud
6. Batch prediction & comparison with kernel baseline
7. Full evaluation (NLL, MSE, per-horizon breakdown)
8. Bimodal demo: show that branches are preserved


---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is importable
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BUILD_DIR = REPO_ROOT / "morphseq_playground" / "metadata" / "build06_output"
print(f"Repo root : {REPO_ROOT}")
print(f"Build dir : {BUILD_DIR}")
print(f"Available : {BUILD_DIR.exists()}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Ellipse

# Data pipeline
from dev.dynamo.data.loading import load_trajectories
from dev.dynamo.data.dataset import FragmentDataset, fragment_collate_fn

# Models
from dev.dynamo.models.particle_filter import BranchingParticleFilter, _linear_fit
from dev.dynamo.models.kernel import SimpleKernelPredictor

# Eval
from dev.dynamo.eval.evaluate import run_evaluation
from dev.dynamo.eval.wandb_logger import print_eval_summary
from dev.dynamo.viz.panels import prediction_fan, trajectory_view

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
print("Imports OK")

---
## 2. Load data

We use the same `load_trajectories` call as the main sandbox.
If the real build06 data is not available this cell will fall through to a
synthetic dataset that is sufficient for all demonstrations below.

In [ ]:
if BUILD_DIR.exists():
    traj_ds = load_trajectories(BUILD_DIR, n_components=10, verbose=True)
    print(f"Loaded {len(traj_ds.trajectories)} trajectories, "
          f"{len(traj_ds.class_names)} classes, "
          f"{traj_ds.n_components} PC dims")
    USING_REAL = True
else:
    # ---------- synthetic fallback ----------
    print("Build dir not found — using synthetic dataset.")
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    from dev.dynamo.data.loading import EmbryoTrajectory, TrajectoryDataset

    def _synth_traj(embryo_id, n_frames=40, n_dim=5, direction=None, seed=0, noise=0.05, delta_t=300.0, cls="wt", exp="exp_001"):
        rng = np.random.default_rng(seed)
        if direction is not None:
            t = np.linspace(0, 1, n_frames)[:, None]
            traj = t * direction[None] + rng.standard_normal((n_frames, n_dim)) * noise
        else:
            traj = np.cumsum(rng.standard_normal((n_frames, n_dim)) * 0.1, axis=0)
        return EmbryoTrajectory(
            embryo_id=embryo_id, trajectory=traj,
            time_seconds=np.arange(n_frames) * delta_t,
            delta_t=delta_t, temperature=28.5,
            perturbation_class=cls, experiment_id=exp,
        )

    N_DIM = 5
    direction = np.array([1., 0.5, -0.3, 0.1, 0.])
    trajs = [
        _synth_traj(f"e{i}", n_frames=40, n_dim=N_DIM, direction=direction,
                    seed=i, noise=0.03,
                    cls=["wt", "mut_a", "mut_b"][i % 3])
        for i in range(60)
    ]
    pca = PCA(n_components=N_DIM); pca.fit(np.eye(N_DIM))
    scaler = StandardScaler(); scaler.fit(np.zeros((2, N_DIM)))
    traj_ds = TrajectoryDataset(
        trajectories=trajs, pca=pca, scaler=scaler,
        z_mu_cols=[f"z_{i}" for i in range(N_DIM)],
    )
    USING_REAL = False
    print(f"Synthetic: {len(traj_ds.trajectories)} trajectories, "
          f"{len(traj_ds.class_names)} classes, dim={traj_ds.n_components}")

---
## 3. Instantiate the particle filter

Key hyperparameters and their spec references:

| Parameter | Spec symbol | Role |
|-----------|-------------|------|
| `radius` | $R$ | Spatial filter radius for selection and recruitment |
| `sigma_pos` | $\sigma_{\text{pos}}$ | Positional weight bandwidth |
| `directional_alpha` | $\alpha$ | Directional sensitivity exponent |
| `sigma_recruit` | $\sigma$ | Recruitment kernel bandwidth |
| `n_max_particles` | $N_{\max}$ | Hard cap on active particles |
| `context_window` | $W$ | Recent frames used for query direction fit |


In [ ]:
# Instantiate with auto-calibrated radius.
pf = BranchingParticleFilter(
    traj_ds,
    radius=None,            # auto via median pairwise distance
    min_points_in_radius=3,
    context_window=7,
    sigma_pos=None,         # auto = radius / 2
    directional_alpha=2.0,
    sigma_recruit=None,     # auto = sigma_pos
    n_max_particles=200,
    n_samples=100,
    exclude_self=True,
)
print(f"radius       = {pf.radius:.4f}")
print(f"sigma_pos    = {pf.sigma_pos:.4f}")
print(f"sigma_recruit= {pf.sigma_recruit:.4f}")
print(f"n_particles  = {pf.n_max_particles}")

---
## 4. Layer 1: reference selection

Pick one embryo, extract its first 7 frames as a context window, and
call `select_references` to see which training trajectories are selected
and what weights they receive.

In [ ]:
# Pick query embryo (use index 0; exclude_self will hide it from references).
QUERY_IDX = 0
query_traj = traj_ds.trajectories[QUERY_IDX]
CTX_LEN = 7
context = query_traj.trajectory[:CTX_LEN]          # (7, D)
delta_t = query_traj.delta_t

particles = pf.select_references(
    context=context,
    delta_t=delta_t,
    embryo_idx=QUERY_IDX,
)
print(f"References selected: {len(particles)}")
if particles:
    weights = np.array([p.weight for p in particles])
    speed_ratios = np.array([p.speed_ratio for p in particles])
    print(f"Weight range    : [{weights.min():.4f}, {weights.max():.4f}]  (sum={weights.sum():.4f})")
    print(f"Speed ratio range: [{speed_ratios.min():.3f}, {speed_ratios.max():.3f}]")
    # Top-5 by weight
    top_idx = np.argsort(weights)[::-1][:5]
    print("\nTop-5 references by weight:")
    print(f"  {'traj_id':>8}  {'anchor':>7}  {'speed_ratio':>12}  {'weight':>10}")
    for i in top_idx:
        p = particles[i]
        print(f"  {p.traj_id:>8}  {p.anchor_frame:>7}  {p.speed_ratio:>12.4f}  {p.weight:>10.6f}")

In [ ]:
# Visualise reference weights in PC1-PC2 space.
if particles:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: PC1 vs PC2 of all training trajectories, coloured by weight.
    ax = axes[0]
    # All anchor positions.
    anchor_pts = np.array([
        traj_ds.trajectories[p.traj_id].trajectory[p.anchor_frame]
        for p in particles
    ])  # (N_ref, D)
    sc = ax.scatter(
        anchor_pts[:, 0], anchor_pts[:, 1],
        c=[p.weight for p in particles],
        cmap="viridis", s=60, zorder=3, label="reference anchors",
    )
    plt.colorbar(sc, ax=ax, label="weight")
    # Query context.
    ax.plot(context[:, 0], context[:, 1], "r-o", lw=2, ms=5, label="query context", zorder=4)
    ax.plot(context[-1, 0], context[-1, 1], "r*", ms=14, zorder=5, label="query point")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.set_title("Reference anchor weights (PC1-PC2)")
    ax.legend(fontsize=8)

    # Right: weight vs speed_ratio.
    ax = axes[1]
    ax.scatter([p.speed_ratio for p in particles], [p.weight for p in particles],
               alpha=0.7, s=40)
    ax.axvline(1.0, ls="--", color="gray", label="speed_ratio=1")
    ax.set_xlabel("Speed ratio ρ (v_query / v_ref)")
    ax.set_ylabel("Normalised weight")
    ax.set_title("Weight vs speed ratio")
    ax.legend()

    fig.tight_layout()
    plt.show()

---
## 5. Layer 2+3: forward prediction — single example

Run `_forward_predict` explicitly so we can inspect how the particle set
evolves across steps.  We run **with** and **without** recruitment to show
the effect.

In [ ]:
if particles:
    import copy

    delta_t_q = query_traj.delta_t
    N_STEPS = 4

    # Query speed.
    diffs = np.linalg.norm(np.diff(context, axis=0), axis=1)
    query_speed = float(diffs.mean() / max(delta_t_q, 1e-6))

    # ---- With recruitment (default pf) ----
    p_init = copy.deepcopy(particles)
    p_final_with = pf._forward_predict(
        particles=p_init,
        n_steps=N_STEPS,
        delta_t=delta_t_q,
        embryo_idx=QUERY_IDX,
        query_speed=query_speed,
    )

    # ---- Without recruitment (sigma_recruit → 0) ----
    pf_no_recruit = BranchingParticleFilter(
        traj_ds,
        radius=pf.radius,
        sigma_pos=pf.sigma_pos,
        sigma_recruit=1e-12,   # effectively no recruitment
        directional_alpha=pf.directional_alpha,
        n_max_particles=pf.n_max_particles,
        n_samples=pf.n_samples,
        exclude_self=pf.exclude_self,
    )
    p_init2 = copy.deepcopy(particles)
    p_final_no = pf_no_recruit._forward_predict(
        particles=p_init2,
        n_steps=N_STEPS,
        delta_t=delta_t_q,
        embryo_idx=QUERY_IDX,
        query_speed=query_speed,
    )

    # Collect final positions.
    def collect(p_list, step, dt):
        pts, ws = [], []
        for p in p_list:
            pos = pf._particle_position(p, step, dt)
            if pos is not None:
                pts.append(pos); ws.append(p.weight)
        if not pts:
            return None, None
        return np.array(pts), np.array(ws)

    pts_with, ws_with = collect(p_final_with, N_STEPS, delta_t_q)
    pts_no,   ws_no   = collect(p_final_no,   N_STEPS, delta_t_q)

    print(f"Initial particles : {len(particles)}")
    print(f"With recruitment  : {len(p_final_with)} particles alive at step {N_STEPS}")
    print(f"No recruitment    : {len(p_final_no)} particles alive at step {N_STEPS}")

In [ ]:
if particles and pts_with is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    horizon_dt = N_STEPS * delta_t_q
    true_target_idx = min(CTX_LEN + N_STEPS - 1, len(query_traj.trajectory) - 1)
    true_target = query_traj.trajectory[true_target_idx]  # best-available true target

    for ax, pts, ws, label in [
        (axes[0], pts_with, ws_with, f"With recruitment  (n={len(pts_with)})"),
        (axes[1], pts_no,   ws_no,   f"No recruitment    (n={len(pts_no) if pts_no is not None else 0})"),
    ]:
        if pts is None:
            ax.set_title("No particles")
            continue
        w_norm = ws / ws.sum()
        ax.scatter(pts[:, 0], pts[:, 1], c=w_norm, cmap="plasma",
                   alpha=0.7, s=40, label="particle cloud")
        mean = w_norm @ pts
        ax.plot(*mean[:2], "k*", ms=12, label="weighted mean")
        ax.plot(context[:, 0], context[:, 1], "b-o", lw=2, ms=4, label="query context")
        ax.plot(*true_target[:2], "g^", ms=10, label="true target")
        ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
        ax.set_title(label)
        ax.legend(fontsize=7)

    fig.suptitle(f"Forward prediction at step {N_STEPS} ({horizon_dt:.0f}s horizon)", fontsize=12)
    fig.tight_layout()
    plt.show()

---
## 6. Prediction fan

Use the standard `prediction_fan` panel from `viz/panels.py` for a batch
of 4 query embryos side by side.

In [ ]:
# Build a small batch.
N_BATCH = 4
fds = FragmentDataset(traj_ds, min_context=5, max_context=10, horizons=(2, 3, 4))
samples = [fds[i] for i in range(N_BATCH)]
batch = fragment_collate_fn(samples)

print("Batch shapes:")
print(f"  context  : {batch.context.shape}")
print(f"  target   : {batch.target.shape}")
print(f"  horizon_dt: {batch.horizon_dt.tolist()}")

In [ ]:
# Run the particle filter on this batch.
pf_result = pf.predict(batch)
print("Prediction result:")
print(f"  mean shape    : {pf_result.predicted_mean.shape}")
print(f"  cov_diag shape: {pf_result.predicted_cov_diag.shape}")
print(f"  samples shape : {pf_result.forward_samples.shape}")
print(f"  mean cov_diag : {pf_result.predicted_cov_diag.mean():.4f}")

In [ ]:
# Draw a 4-panel prediction fan (one per batch sample).
fig, axes = plt.subplots(1, N_BATCH, figsize=(4 * N_BATCH, 4))
if N_BATCH == 1:
    axes = [axes]

for b, ax in enumerate(axes):
    L = int(batch.context_mask[b].sum())
    ctx = batch.context[b, :L].numpy()     # (L, D)
    tgt = batch.target[b].numpy()          # (D,)
    mean = pf_result.predicted_mean[b].numpy()
    samps = pf_result.forward_samples[b].numpy()  # (n_samples, D)

    # Background: grey scatter of all training points (PC1-PC2).
    all_pts = pf.bank.all_points
    ax.scatter(all_pts[::5, 0], all_pts[::5, 1], c="lightgrey", s=2, zorder=0)

    # Particle cloud, sized & coloured by (equal) weight.
    ax.scatter(samps[:, 0], samps[:, 1], c="steelblue", alpha=0.4, s=12,
               label="samples", zorder=2)

    # Context trajectory.
    ax.plot(ctx[:, 0], ctx[:, 1], "k-o", lw=1.5, ms=3, label="context", zorder=3)

    # Predicted mean.
    ax.plot(*mean[:2], "b*", ms=12, label="pred mean", zorder=4)

    # True target.
    ax.plot(*tgt[:2], "r^", ms=10, label="true target", zorder=5)

    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.set_title(f"Sample {b}  |  horizon={batch.horizon_dt[b]:.0f}s")
    if b == 0:
        ax.legend(fontsize=7)

fig.suptitle("Branching particle filter — prediction fan", fontsize=12)
fig.tight_layout()
plt.show()

---
## 7. Comparison with simple kernel baseline

Run both models on the same batch and compare mean predictions.

In [ ]:
kernel = SimpleKernelPredictor(traj_ds, bandwidth=None, n_samples=100, exclude_self=True)
kernel_result = kernel.predict(batch)

# Per-sample MSE comparison.
pf_mse   = (pf_result.predicted_mean - batch.target).pow(2).mean(dim=-1)    # (B,)
kern_mse = (kernel_result.predicted_mean - batch.target).pow(2).mean(dim=-1)

print(f"{'Sample':>6}  {'PF MSE':>9}  {'Kernel MSE':>10}  {'winner':>8}")
for b in range(N_BATCH):
    winner = "PF" if pf_mse[b] < kern_mse[b] else "kernel"
    print(f"{b:>6}  {pf_mse[b].item():>9.4f}  {kern_mse[b].item():>10.4f}  {winner:>8}")

print(f"\nMean  {pf_mse.mean().item():>9.4f}  {kern_mse.mean().item():>10.4f}")

In [ ]:
# Side-by-side prediction fans for the first sample.
b = 0
L = int(batch.context_mask[b].sum())
ctx = batch.context[b, :L].numpy()
tgt = batch.target[b].numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, result, title in [
    (ax1, pf_result,     "Branching particle filter"),
    (ax2, kernel_result, "Simple kernel regression"),
]:
    samps = result.forward_samples[b].numpy()
    mean  = result.predicted_mean[b].numpy()
    ax.scatter(samps[:, 0], samps[:, 1], c="steelblue", alpha=0.35, s=10)
    ax.plot(ctx[:, 0], ctx[:, 1], "k-o", lw=1.5, ms=3, label="context")
    ax.plot(*mean[:2], "b*", ms=13, label="pred mean")
    ax.plot(*tgt[:2], "r^", ms=11, label="true target")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.suptitle(f"Sample 0  |  horizon={batch.horizon_dt[0]:.0f}s", fontsize=11)
fig.tight_layout()
plt.show()

---
## 8. Full evaluation — NLL, MSE, per-horizon breakdown

`run_evaluation` runs the predictor over many batches and aggregates
metrics using the standard eval pipeline.

In [ ]:
N_BATCHES = 20   # Increase for more accurate estimates (but slower).
BATCH_SIZE = 16

eval_fds = FragmentDataset(traj_ds, min_context=5, max_context=12, horizons=(1, 2, 3, 4))

print("Evaluating particle filter...")
pf_eval = run_evaluation(pf, eval_fds, n_batches=N_BATCHES, batch_size=BATCH_SIZE)
print()

print("Evaluating kernel baseline...")
kern_eval = run_evaluation(kernel, eval_fds, n_batches=N_BATCHES, batch_size=BATCH_SIZE)
print()

print("=" * 50)
print("Particle filter:")
print_eval_summary(pf_eval)
print()
print("Simple kernel:")
print_eval_summary(kern_eval)

In [ ]:
# Per-horizon NLL and MSE comparison.
horizons = sorted(pf_eval.per_horizon.keys())

if horizons:
    pf_nll  = [pf_eval.per_horizon[k].get("nll", float("nan"))  for k in horizons]
    pf_mse  = [pf_eval.per_horizon[k].get("mse", float("nan"))  for k in horizons]
    kn_nll  = [kern_eval.per_horizon[k].get("nll", float("nan")) for k in horizons]
    kn_mse  = [kern_eval.per_horizon[k].get("mse", float("nan")) for k in horizons]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(horizons, pf_nll,  "b-o", label="Particle filter")
    ax1.plot(horizons, kn_nll,  "r--s", label="Kernel")
    ax1.set_xlabel("Horizon k (frames)")
    ax1.set_ylabel("NLL")
    ax1.set_title("Per-horizon NLL")
    ax1.legend()

    ax2.plot(horizons, pf_mse,  "b-o", label="Particle filter")
    ax2.plot(horizons, kn_mse,  "r--s", label="Kernel")
    ax2.set_xlabel("Horizon k (frames)")
    ax2.set_ylabel("MSE")
    ax2.set_title("Per-horizon MSE")
    ax2.legend()

    fig.tight_layout()
    plt.show()
else:
    print("No per-horizon breakdown available (all horizons in same bucket).")

---
## 9. Bimodal demo

Build a synthetic dataset where trajectories split into two branches after
a shared origin.  Show that:

- The **kernel baseline** smears probability across both branches → unimodal
  prediction near the bifurcation point.
- The **branching particle filter** maintains two independent particle pools
  → bimodal prediction.

This is the key property from spec §9.2.2 "Branch preservation".

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from dev.dynamo.data.loading import EmbryoTrajectory, TrajectoryDataset

D_BI = 2
N_PER_BRANCH = 15
N_FRAMES_BI = 30

def _bi_traj(embryo_id, direction, seed, n_frames=N_FRAMES_BI, noise=0.003):
    rng = np.random.default_rng(seed)
    start = rng.standard_normal(D_BI) * 0.005
    t = np.linspace(0, 1, n_frames)[:, None]
    traj = start[None] + t * direction[None] + rng.standard_normal((n_frames, D_BI)) * noise
    return EmbryoTrajectory(
        embryo_id=embryo_id,
        trajectory=traj, time_seconds=np.arange(n_frames, dtype=float) * 300.,
        delta_t=300., temperature=28.5, perturbation_class="branch",
        experiment_id="exp_bi",
    )

# Branch A goes +y, branch B goes -y.
bi_trajs = (
    [_bi_traj(f"a{i}", np.array([1., 0.8]), seed=i, noise=0.002)   for i in range(N_PER_BRANCH)] +
    [_bi_traj(f"b{i}", np.array([1., -0.8]), seed=i+100, noise=0.002) for i in range(N_PER_BRANCH)]
)

pca_bi = PCA(n_components=D_BI); pca_bi.fit(np.eye(D_BI))
sc_bi  = StandardScaler(); sc_bi.fit(np.zeros((2, D_BI)))
bi_ds = TrajectoryDataset(
    trajectories=bi_trajs, pca=pca_bi, scaler=sc_bi,
    z_mu_cols=[f"z_{i}" for i in range(D_BI)],
)

print(f"Bifurcating dataset: {len(bi_ds.trajectories)} trajectories")
# Quick sanity: show endpoint spread.
ends = np.array([t.trajectory[-1] for t in bi_ds.trajectories])
print(f"  endpoint y-range: [{ends[:, 1].min():.3f}, {ends[:, 1].max():.3f}]")

In [ ]:
# Build a particle filter and kernel on this bifurcating dataset.
bi_pf = BranchingParticleFilter(
    bi_ds, radius=0.5, min_points_in_radius=1,
    sigma_pos=0.25, sigma_recruit=0.25,
    n_max_particles=300, n_samples=200, exclude_self=False,
)
bi_kernel = SimpleKernelPredictor(bi_ds, bandwidth=0.5, n_samples=200, exclude_self=False)

# Query: short horizontal context (pre-bifurcation).
context_bi = np.column_stack([np.linspace(0, 0.08, 5), np.zeros(5)])
delta_t_bi = 300.0
HORIZON_STEPS = 6

pf_single = bi_pf._predict_single(
    context=context_bi, delta_t=delta_t_bi,
    horizon_dt=HORIZON_STEPS * delta_t_bi, embryo_idx=-1,
)

# For kernel: build a minimal batch.
from dev.dynamo.data.dataset import FragmentBatch
import torch

L = len(context_bi)
_ctx_t = torch.from_numpy(context_bi).float().unsqueeze(0)  # (1, L, D)
_mask  = torch.ones(1, L, dtype=torch.bool)
_tgt   = torch.zeros(1, D_BI)
_td    = torch.zeros(1, L - 1)
_hdt   = torch.tensor([HORIZON_STEPS * delta_t_bi], dtype=torch.float32)
_dt    = torch.tensor([delta_t_bi], dtype=torch.float32)

bi_batch = FragmentBatch(
    context=_ctx_t, context_mask=_mask, target=_tgt,
    time_deltas=_td, horizon_dt=_hdt, delta_t=_dt,
    temperature=torch.tensor([28.5]), class_idx=torch.tensor([0]),
    embryo_idx=torch.tensor([0]),
)
kern_single_result = bi_kernel.predict(bi_batch)
kern_samps = kern_single_result.forward_samples[0].numpy()

pf_samps = pf_single["samples"]   # (200, D)

print(f"PF samples  y std  : {pf_samps[:, 1].std():.4f}")
print(f"Kernel samps y std : {kern_samps[:, 1].std():.4f}")

In [ ]:
# Visualise both predictive distributions on the bifurcating data.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: dataset overview.
ax = axes[0]
colors = ["tab:blue"] * N_PER_BRANCH + ["tab:orange"] * N_PER_BRANCH
for i, traj in enumerate(bi_trajs):
    ax.plot(traj.trajectory[:, 0], traj.trajectory[:, 1],
            color=colors[i], alpha=0.5, lw=1.2)
ax.plot(context_bi[:, 0], context_bi[:, 1], "k-o", lw=2, ms=5, label="query context")
ax.plot(*context_bi[-1], "k*", ms=12)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.set_title("Bifurcating training data")
ax.legend(fontsize=8)

# Middle: particle filter cloud.
ax = axes[1]
ax.scatter(pf_samps[:, 0], pf_samps[:, 1], c="steelblue", alpha=0.4, s=15,
           label=f"PF (y_std={pf_samps[:,1].std():.3f})")
ax.plot(context_bi[:, 0], context_bi[:, 1], "k-o", lw=2, ms=4)
ax.axhline(0, ls="--", color="gray", alpha=0.5)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.set_title("Particle filter — bimodal")
ax.legend(fontsize=8)

# Right: kernel cloud.
ax = axes[2]
ax.scatter(kern_samps[:, 0], kern_samps[:, 1], c="tomato", alpha=0.4, s=15,
           label=f"Kernel (y_std={kern_samps[:,1].std():.3f})")
ax.plot(context_bi[:, 0], context_bi[:, 1], "k-o", lw=2, ms=4)
ax.axhline(0, ls="--", color="gray", alpha=0.5)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.set_title("Kernel — smeared")
ax.legend(fontsize=8)

fig.suptitle(
    f"Bifurcation demo  |  horizon={HORIZON_STEPS} steps  |  "
    "PF preserves branches, kernel smears them",
    fontsize=11,
)
fig.tight_layout()
plt.show()

In [ ]:
# Histogram of y-coordinate: both models side by side.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)

ax1.hist(pf_samps[:, 1], bins=30, color="steelblue", edgecolor="white")
ax1.set_xlabel("y (PC 2)")
ax1.set_title(f"Particle filter  (std={pf_samps[:,1].std():.3f})")
ax1.axvline(0, ls="--", color="gray")

ax2.hist(kern_samps[:, 1], bins=30, color="tomato", edgecolor="white")
ax2.set_xlabel("y (PC 2)")
ax2.set_title(f"Kernel  (std={kern_samps[:,1].std():.3f})")
ax2.axvline(0, ls="--", color="gray")

fig.suptitle("Distribution of y-coordinate at bifurcation (bimodal vs smeared)", fontsize=11)
fig.tight_layout()
plt.show()

---
## 10. Summary

The cells above demonstrate the full three-layer implementation:

| Layer | What it does | Code entry point |
|-------|-------------|------------------|
| **1 — Selection** | Direction-aware reference matching with speed ratios | `pf.select_references(context, delta_t)` |
| **2 — Tracking** | Advance particles by `ρ·k·dt_query/dt_ref` frames; drop dead ones | `pf._particle_position(p, step, dt)` |
| **3 — Recruitment** | Each live particle recruits nearby trajectories, inheriting weight | `pf._recruit_from(position, parent_weight, ...)` |

Key hyperparameters to tune on real data:
- **`radius`** — start at auto (median pairwise distance), reduce if too many references.
- **`directional_alpha`** — increase (3–5) for stronger direction filtering.
- **`sigma_recruit`** — tighten if recruitment is too aggressive.
- **`n_max_particles`** — 200 is enough for most queries; increase for bifurcation-heavy data.

Next build step: **phi0-only model** (Stage 1 learned potential).